# 6.1 Introduction to mapping

In the previous chapter we studied different methods for localizing a robot equipped with a sensor able to take measurements to a number of landmarks with known positions, that is, the map $\mathbf{m}$ of the environment was given beforehand. Now we will review a number of techniques addressing the opposite problem: to build a map $\mathbf{m}$ of the environment given a set of known robot poses $\mathbf{x}_{1:t}$ and sensor measurements $\mathbf{z}_{1:t}$, that is:

$$p(\mathbf{m} | \mathbf{z}_{1:t},\mathbf{x}_{1:t})$$

Notice that:

- the poses $\mathbf{x}_{1:t}$ are not random variables here, but known vectors ($3\times1$ each in 2D).
- since the robot poses are given, it is not necessary to consider the motion commands $\mathbf{u}_{1:t}$.
- each observation $\mathbf{z}_t$ is a sample from a distribution with mean $\mathbf{h}(\mathbf{x}_t,\mathbf{m})$ (recall the observation function $\mathbf{h}$ from notebook 4.1).

There exist different types of maps, being the most used ones:
<table>
    <tr>
        <td style="text-align: center; vertical-align: middle;"><b>Landmarks based maps</b></td>
        <td style="text-align: center; vertical-align: middle;"><b>Occupancy grid maps</b></td>
    </tr>
    <tr>
        <td style="text-align: center; vertical-align: middle;">
            <figure>
              <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/mapping/images/map_landmarks.PNG" />
              <figcaption>Fig. 1. Landmarks based map of an office environment. In this case landmarks are QR codes placed next to doors.</figcaption>        
            </figure>
        </td>
        <td style="text-align: center; vertical-align: middle;">
             <figure>
              <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/mapping/images/map_occupancy_gridmap.png"/>
              <figcaption>Fig. 2. Example of an occupancy gridmap of a house. In the image, white pixels represent traversable space, black pixels stand for obstacles, and gray ones are unknown space.</figcaption>
            </figure>
        </td>
    </tr>
    <tr>
        <td style="text-align: center; vertical-align: middle;"><b>Topological maps</b></td>
        <td style="text-align: center; vertical-align: middle;"><b>Semantic maps</b></td>
    </tr>
    <tr>
        <td style="text-align: center; vertical-align: middle;">
            <figure>
              <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/mapping/images/map_topological.png" width="250"/>
              <figcaption>Fig. 3. Example of a topological map of a house where nodes represent rooms and edges link connected rooms.</figcaption>
            </figure>
        </td>
        <td style="text-align: center; vertical-align: middle;">
            <figure>
              <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/mapping/images/map_semantic.png" width="450"/>
              <figcaption>Fig. 4. Example of a semantic map of a bathroom where the objects within it are linked with meta-information about their category, size, utility, mobility, etc.</figcaption>
            </figure>
        </td>
    </tr>
</table>

<br />

Notice that both Landmarks based maps and occupancy grid maps are **geometric maps**. In the next section of this book we are going to take a look at how to build landmarks based maps relying on the Extended Kalman Filter (EKF).

## <span style="color:green">OPTIONAL</span>

<span style="color:green">Surf the internet looking for more general information about robot mapping. You can include additional definition, examples, images, videos,... anything you find interesting!</span>

<div style="color:#2DD4BF">

### 1. Por qué es difícil construir un mapa

Thrun ([*Robotic Mapping: A Survey*, 2002](http://robots.stanford.edu/papers/thrun.mapping-tr.pdf)) resume en cinco los problemas que hacen difícil el mapeado:

1. **Ruido en las medidas**, que además no es independiente: los errores de pose se acumulan y afectan a cómo se interpretan las medidas siguientes.
2. **Alta dimensionalidad**: un mapa 2D de un edificio puede necesitar miles de números (landmarks o celdas), y uno 3D, millones.
3. **El problema de correspondencia (asociación de datos)**: decidir si dos medidas tomadas en momentos distintos corresponden al mismo objeto. Según Thrun, posiblemente el más difícil.
4. **Entornos que cambian**: puertas, muebles o personas en movimiento.
5. **Exploración**: el robot tiene que decidir por dónde moverse para completar el mapa.

En este capítulo se simplifica el problema al suponer que **las poses $\mathbf{x}_{1:t}$ son conocidas**. Así desaparece el ruido de pose acumulado y queda $p(\mathbf{m} \mid \mathbf{z}_{1:t}, \mathbf{x}_{1:t})$. Si las poses también hay que estimarlas, el problema pasa a ser **SLAM** (*Simultaneous Localization And Mapping*): $p(\mathbf{x}_{1:t}, \mathbf{m} \mid \mathbf{z}_{1:t}, \mathbf{u}_{1:t})$.

### 2. Comparación de los cuatro tipos de mapa

| Tipo | Qué guarda | Memoria | Para qué sirve mejor | Ejemplo |
| --- | --- | --- | --- | --- |
| **Landmarks** | Posición (y covarianza) de cada marca reconocible | Muy poca: crece con el número de marcas | Localización con EKF (práctica 5.2) y SLAM con EKF | Códigos QR junto a las puertas |
| **Rejilla de ocupación** | $P(\text{ocupada})$ de cada celda | Crece con el área / resolución²: 14 × 12 m a 0,1 m son 16 800 celdas | Navegación, evitar obstáculos, planificación, localización con láser (AMCL, 5.3 y 5.4) | Mapa `.pgm` + `.yaml` de Nav2 |
| **Topológico** | Lugares (nodos) y conexiones transitables (aristas), sin geometría precisa | Mínima | Planificación a gran escala («cocina → pasillo → salón») | Grafo de habitaciones de una casa |
| **Semántico** | Objetos y lugares con categoría y propiedades (uso, tamaño, si se pueden mover…) | Variable; normalmente encima de un mapa geométrico | Tareas que se expresan como lo haría una persona («trae la taza de la cocina») | Baño con lavabo, toalla y ducha etiquetados |

Los dos primeros son **mapas geométricos** (métricos). En la práctica se combinan: Kuipers propuso con la *[Spatial Semantic Hierarchy](https://doi.org/10.1016/S0004-3702(00)00017-5)* (2000) que la topología sea la base y que la geometría se vaya añadiendo después. Kostavelis y Gasteratos ([*Semantic mapping for mobile robotics tasks: A survey*, 2015](https://doi.org/10.1016/j.robot.2014.12.006)) señalan que un mapa métrico perfecto no basta para que el robot entienda el entorno como una persona: tiene que distinguir, por ejemplo, una habitación de un pasillo.

### 3. Origen de las rejillas de ocupación

Las rejillas de ocupación las introdujeron Moravec y Elfes a mediados de los 80. Elfes las describe en [*Using occupancy grids for mobile robot perception and navigation*](https://www.cs.cmu.edu/~motionplanning/papers/sbp_papers/integrated4/elfes_occup_grids.pdf) (IEEE Computer, 1989) como un **campo aleatorio binario**: cada celda tiene dos estados, ocupada o vacía, cuyas probabilidades suman 1. Esas estimaciones se actualizan con Bayes a partir de medidas de rango interpretadas con **modelos probabilísticos del sensor**. Es exactamente lo que se implementa en el notebook 6.3.

### 4. Del 2D al 3D: OctoMap

Para robots que se mueven en 3D (drones, brazos, robots en exteriores), una rejilla 3D uniforme gastaría muchísima memoria. [OctoMap](https://octomap.github.io/) (Hornung et al., [*Autonomous Robots*, 2013](https://doi.org/10.1007/s10514-012-9321-0)) guarda la ocupación en un **octree**: cada nodo es un cubo (vóxel) que se divide en 8 hasta la resolución mínima.

- **Mismo modelo que la 6.3:** los vóxeles se actualizan con **log odds**, y el espacio libre se obtiene trazando rayos desde el sensor hasta el punto medido.
- **Libre, ocupado y desconocido:** el espacio libre se guarda explícitamente y lo desconocido son los nodos que no existen. Es importante para explorar.
- **Clamping:** el log odds se limita entre $l_{min} = -2$ y $l_{max} = 3{,}5$ (probabilidades 0,12 y 0,97). Así el mapa se adapta rápido si el entorno cambia, y los nodos hijos con el mismo estado se pueden **podar** para comprimir.
- **Modelo de sensor:** usa 0,7 para un impacto y 0,4 para un vóxel atravesado. En la 6.3 son 0,8 y 0,2.
- **Multirresolución:** el árbol se puede cortar a cualquier nivel. Un planificador global puede usar un mapa grueso y uno local, el fino.

<figure style="text-align:center">
  <img src="https://octomap.github.io/freiburg_079.png" width="600" alt="OctoMap del edificio 079 de la Universidad de Friburgo">
  <figcaption>Mapa 3D con OctoMap de un pasillo del edificio 079 de la Universidad de Friburgo (imagen de <a href="https://octomap.github.io/" target="_blank">octomap.github.io</a>). Vídeos: <a href="https://www.youtube.com/watch?v=O2TDNJuHMKo" target="_blank">edificio 079</a> y <a href="https://www.youtube.com/watch?v=7ZsxJzR14rc" target="_blank">campus de Friburgo</a>.</figcaption>
</figure>

### 5. Mapas de rejilla en ROS 2

Es el formato que ya se ha usado con AMCL en la práctica 5.4:

- El mensaje `nav_msgs/msg/OccupancyGrid` guarda las celdas por filas, empezando en la (0, 0). Cada celda tiene un valor de 0 (libre) a 100 (ocupada), o −1 si es desconocida. El orden por filas es el mismo `[y][x]` que hay que respetar en la 6.3.
- El [map server de Nav2](https://docs.nav2.org/rolling/configuration_and_development/configuration_guide/core_servers/map_server/configuring_map_server/) carga el mapa desde una imagen (`.pgm`/`.png`) más un `.yaml` con `resolution`, `origin`, `occupied_thresh` (típicamente 0,65) y `free_thresh` (típicamente 0,196). En el modo por defecto (`trinary`), cada píxel se clasifica como ocupado, libre o desconocido con esos dos umbrales, igual que los umbrales 0,7 / 0,3 del final de la 6.3.
- Los mapas se suelen construir con SLAM, por ejemplo con [SLAM Toolbox](https://github.com/SteveMacenski/slam_toolbox), la librería de SLAM 2D con soporte oficial en ROS 2. Después se guardan en ese formato y se localiza sobre ellos con AMCL.

### 6. Idea clave

Elegir el tipo de mapa es elegir **qué pregunta tiene que poder responder el robot**:

- «¿Dónde estoy?»: landmarks o rejilla.
- «¿Puedo pasar por aquí?»: rejilla u OctoMap.
- «¿Cómo llego a la cocina?»: topológico.
- «¿Dónde suele estar la taza?»: semántico.

Los robots reales suelen combinar varios.

</div>

<span style="color:green">***END OF OPTIONAL PART***</span>